# Klasifikacija motocikala — Seminarski rad
**Studentica:** Hasna Bunar, IB220124
**Predmet:** Mašinsko učenje
**Mentorica:** prof. dr. Mirela Mabić
**Akademska godina:** 2025/2026

---
Ovaj notebook implementira kompletan pipeline mašinskog učenja za klasifikaciju motocikala u 18 kategorija na osnovu tehničkih specifikacija iz dataseta Bikez.com (38.472 motocikla, 85 varijabli).


# 1. Importing libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import re
import json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    adjusted_rand_score, silhouette_score,
    davies_bouldin_score, calinski_harabasz_score
)
from kmodes.kmodes import KModes
from kmodes.kprototypes import KPrototypes

sns.set_theme(style='whitegrid')
print("Sve biblioteke uspješno učitane.")


: 

# 2. Loading dataset
Dataset je preuzet sa stranice **Bikez.com** i sadrži tehničke specifikacije motocikala objavljenih između 1894. i 2023. godine.
Fajl se nalazi u folderu `data/all_bikez_raw.csv`.


In [ ]:
df_raw = pd.read_csv('data/all_bikez_raw.csv', low_memory=False)
print(f"Shape dataseta: {df_raw.shape}")
df_raw.head(3)


# 3. Data understanding
Ova faza obuhvata analizu strukture dataseta, distribuciju ciljne varijable (Category), analizu nedostajućih vrijednosti i vizualizaciju ključnih varijabli.


## 3.1 Osnovna struktura

In [ ]:
print(f"Broj redova:   {df_raw.shape[0]}")
print(f"Broj kolona:   {df_raw.shape[1]}")
print(f"\nTipovi podataka:")
print(df_raw.dtypes.value_counts())


## 3.2 Distribucija kategorija (ciljna varijabla)

In [ ]:
cat_counts = df_raw['Category'].value_counts()
print(f"Broj kategorija: {cat_counts.shape[0]}")
print(cat_counts)

fig, ax = plt.subplots(figsize=(12, 5))
cat_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Distribucija motocikala po kategorijama', fontsize=13)
ax.set_xlabel('Kategorija')
ax.set_ylabel('Broj motocikala')
ax.tick_params(axis='x', rotation=45)
for bar in ax.patches:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()


## 3.3 Analiza nedostajućih vrijednosti

In [ ]:
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(1)
missing_df = pd.DataFrame({'Nedostaje': missing, 'Postotak (%)': missing_pct})
missing_df = missing_df[missing_df['Nedostaje'] > 0].sort_values('Postotak (%)', ascending=False)

print(f"Kolone s nedostajućim podacima: {len(missing_df)} od {df_raw.shape[1]}")
print(missing_df.head(20).to_string())

# Vizualizacija top 20
fig, ax = plt.subplots(figsize=(12, 5))
missing_df.head(20)['Postotak (%)'].plot(kind='bar', ax=ax, color='coral')
ax.set_title('Top 20 kolona s najviše nedostajućih vrijednosti')
ax.set_ylabel('Postotak nedostajućih (%)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


## 3.4 Deskriptivna statistika ključnih varijabli

In [ ]:
def izvuci_broj(serija, pattern):
    """Izvlači prvi broj koji odgovara regex šablonu iz tekstualne kolone."""
    return serija.astype(str).str.extract(pattern)[0].astype(float)

df = df_raw.copy()

# Parsiranje numeričkih vrijednosti iz teksta (regex ekstrakcija)
# Raw Bikez kolone sadrže tekstualne vrijednosti npr. "49.9 ccm (3.04 cubic inches)"
df['kubikaza_ccm']     = izvuci_broj(df['Displacement'],  r'([\d.]+)\s*ccm')
df['snaga_hp']         = izvuci_broj(df['Power'],         r'([\d.]+)\s*HP')
df['moment_nm']        = izvuci_broj(df['Torque'],        r'([\d.]+)\s*Nm')
df['tezina_kg']        = izvuci_broj(df['Dry weight'],    r'([\d.]+)\s*kg')
df['brzina_kmh']       = izvuci_broj(df['Top speed'],     r'([\d.]+)\s*km/h')
df['rezervoar_l']      = izvuci_broj(df['Fuel capacity'], r'([\d.]+)\s*litres')
df['sjediste_mm']      = izvuci_broj(df['Seat height'],   r'([\d.]+)\s*mm')
df['meduosovinski_mm'] = izvuci_broj(df['Wheelbase'],     r'([\d.]+)\s*mm')
df['ocjena']           = izvuci_broj(df['Rating'],        r'(\d\.\d)')
df['godina']           = pd.to_numeric(df['Year'], errors='coerce')

# Broj brzina iz "6-speed", "5-speed" itd.
if 'Gearbox' in df.columns:
    df['broj_brzina'] = izvuci_broj(df['Gearbox'], r'(\d+)-speed')

# Hlađenje kao kategorijska varijabla
if 'Cooling system' in df.columns:
    df['hladjenje'] = df['Cooling system']

# Greška u podacima: težina 0 kg tretiramo kao NaN
df.loc[df['tezina_kg'] == 0, 'tezina_kg'] = np.nan

num_cols_available = [c for c in ['kubikaza_ccm','snaga_hp','moment_nm','tezina_kg',
                                   'brzina_kmh','rezervoar_l','sjediste_mm',
                                   'meduosovinski_mm','broj_brzina'] if c in df.columns]

print("Parsiranje završeno. Deskriptivna statistika numeričkih varijabli:")
df[num_cols_available].describe().round(2)


## 3.5 Histogrami ključnih varijabli

In [ ]:
plot_cols = [c for c in ['kubikaza_ccm','snaga_hp','tezina_kg','brzina_kmh'] if c in df.columns]
fig, axes = plt.subplots(1, len(plot_cols), figsize=(5*len(plot_cols), 4))
if len(plot_cols) == 1:
    axes = [axes]
labels = {'kubikaza_ccm':'Kubikaza (ccm)','snaga_hp':'Snaga (HP)',
          'tezina_kg':'Težina (kg)','brzina_kmh':'Max brzina (km/h)'}
for ax, col in zip(axes, plot_cols):
    data = df[col].dropna()
    ax.hist(data, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(labels.get(col, col))
    ax.set_xlabel(labels.get(col, col))
    ax.set_ylabel('Frekvencija')
    ax.axvline(data.median(), color='red', ls='--', lw=1.5, label=f'Medijan: {data.median():.0f}')
    ax.legend(fontsize=8)
fig.suptitle('Distribucija ključnih numeričkih varijabli', fontsize=12)
plt.tight_layout()
plt.show()


## 3.6 Korelacijska matrica

In [ ]:
corr_cols = [c for c in num_cols_available if c in df.columns]
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Korelacijska matrica numeričkih varijabli')
plt.tight_layout()
plt.show()

# Multikolinearnost: snaga_po_kg
if 'snaga_hp' in df.columns and 'tezina_kg' in df.columns:
    valid = df[['snaga_hp','tezina_kg']].dropna()
    snaga_po_kg = valid['snaga_hp'] / valid['tezina_kg']
    print(f"\nKorelacija snaga_hp s izvedenom varijablom snaga_po_kg: {valid['snaga_hp'].corr(snaga_po_kg):.3f}")
    print("→ snaga_po_kg je izvedena varijabla (snaga/težina) — uklanja se iz modela zbog multikolinearnosti.")


# 4. Data preprocessing
Preprocesiranje obuhvata: čišćenje dataseta, feature engineering (razdvajanje kolone Model), dopunu maksimalne brzine i pripremu podataka za trening modela.


## 4.1 Data cleaning — odabir relevantnih varijabli

In [ ]:
# Zadržavamo samo relevantne kolone
keep_cols = ['Category', 'Model'] + num_cols_available
if 'hladjenje' in df.columns:
    keep_cols.append('hladjenje')
if 'dvotaktni_flag' in df.columns:
    keep_cols.append('dvotaktni_flag')
if 'ocjena' in df.columns:
    keep_cols.append('ocjena')

keep_cols = [c for c in keep_cols if c in df.columns]
df = df[keep_cols].copy()
df = df[df['Category'].notna()].reset_index(drop=True)

print(f"Shape nakon čišćenja: {df.shape}")
print(f"Kategorije: {df['Category'].nunique()}")


## 4.2 Feature engineering — razdvajanje kolone Model
Kolona `Model` sadrži brand, naziv modela i ponekad kubikazu u nazivu (npr. "Yamaha YZF-R6 600").
Razdvajamo je u tri zasebne varijable: `brand`, `model_naziv` i `ccm_iz_naziva`.


In [ ]:
def izvuci_brand(naziv):
    if pd.isna(naziv): return np.nan
    return str(naziv).split()[0] if str(naziv).split() else np.nan

def izvuci_ccm(naziv):
    if pd.isna(naziv): return np.nan
    kandidati = re.findall(r'\b(\d{2,4})\b', str(naziv))
    for k in reversed(kandidati):
        val = int(k)
        if 25 <= val <= 2500:
            return float(val)
    return np.nan

if 'Model' in df.columns:
    df['brand'] = df['Model'].apply(izvuci_brand)
    df['model_naziv'] = df['Model'].apply(lambda x: ' '.join(str(x).split()[1:]) if pd.notna(x) else np.nan)
    df['ccm_iz_naziva'] = df['Model'].apply(izvuci_ccm)

    pokriveno = df['ccm_iz_naziva'].notna().mean()
    print(f"ccm_iz_naziva pronađen u {pokriveno:.1%} modela")
    df.drop(columns=['Model'], inplace=True)

df.head(3)


## 4.3 Dopuna maksimalne brzine iz literature
Za najpopularnije modele dodajemo poznatu maksimalnu brzinu iz specifikacija proizvođača.
Ovo povećava pokrivenost varijable `brzina_kmh` i pomaže razlikovanju škutera od sportskih motocikala.


In [ ]:
LOOKUP = [
    # (brand_lower, [kljucne_rijeci], brzina_kmh)
    ("yamaha", ["yzf-r1"], 299), ("yamaha", ["yzf-r6"], 249), ("yamaha", ["yzf-r3"], 180),
    ("yamaha", ["mt-09"], 220), ("yamaha", ["mt-07"], 201), ("yamaha", ["mt-03"], 160),
    ("yamaha", ["nmax"], 125), ("yamaha", ["xmax"], 145), ("yamaha", ["tmax"], 160),
    ("yamaha", ["tracer", "900"], 215), ("yamaha", ["super", "ténéré"], 200),
    ("honda", ["cbr1000rr"], 282), ("honda", ["cbr600rr"], 250), ("honda", ["cbr500r"], 185),
    ("honda", ["cbr300r"], 160), ("honda", ["cb1000r"], 240), ("honda", ["cb650r"], 210),
    ("honda", ["africa", "twin"], 210), ("honda", ["gold", "wing"], 180),
    ("honda", ["pcx"], 120), ("honda", ["sh", "125"], 110), ("honda", ["forza"], 140),
    ("kawasaki", ["ninja", "zx-10r"], 299), ("kawasaki", ["ninja", "zx-6r"], 250),
    ("kawasaki", ["ninja", "400"], 180), ("kawasaki", ["z1000"], 250),
    ("kawasaki", ["z900"], 232), ("kawasaki", ["z650"], 196), ("kawasaki", ["versys", "1000"], 230),
    ("kawasaki", ["h2r"], 400), ("kawasaki", ["h2"], 299),
    ("suzuki", ["hayabusa"], 312), ("suzuki", ["gsx-r1000"], 299), ("suzuki", ["gsx-r750"], 250),
    ("suzuki", ["gsx-r600"], 250), ("suzuki", ["gsx-s1000"], 250), ("suzuki", ["v-strom", "1050"], 220),
    ("bmw", ["s1000rr"], 299), ("bmw", ["s1000r"], 250), ("bmw", ["r1250gs"], 220),
    ("bmw", ["r1250r"], 220), ("bmw", ["f900r"], 210), ("bmw", ["f850gs"], 200),
    ("bmw", ["r ninet"], 200), ("bmw", ["k1600"], 250),
    ("ducati", ["panigale", "v4"], 305), ("ducati", ["panigale", "v2"], 270),
    ("ducati", ["streetfighter", "v4"], 299), ("ducati", ["monster", "1200"], 261),
    ("ducati", ["multistrada", "1260"], 250), ("ducati", ["supersport"], 250),
    ("ducati", ["scrambler"], 170), ("ducati", ["diavel"], 270),
    ("ktm", ["1290", "super duke r"], 290), ("ktm", ["890", "duke"], 225),
    ("ktm", ["790", "duke"], 220), ("ktm", ["390", "duke"], 167),
    ("ktm", ["1290", "super adventure"], 220), ("ktm", ["rc", "390"], 167),
    ("aprilia", ["rsv4"], 299), ("aprilia", ["rs", "660"], 220), ("aprilia", ["tuono", "v4"], 285),
    ("aprilia", ["shiver", "900"], 220), ("aprilia", ["dorsoduro"], 200),
    ("triumph", ["speed", "triple", "1200"], 265), ("triumph", ["street", "triple"], 233),
    ("triumph", ["tiger", "900"], 210), ("triumph", ["bonneville"], 188),
    ("triumph", ["rocket", "3"], 180), ("triumph", ["daytona"], 260),
    ("harley-davidson", ["sportster"], 178), ("harley-davidson", ["fat", "boy"], 160),
    ("harley-davidson", ["road", "king"], 165), ("harley-davidson", ["street", "glide"], 175),
    ("harley-davidson", ["pan america"], 200), ("harley-davidson", ["iron", "883"], 180),
    ("mv agusta", ["f4"], 302), ("mv agusta", ["brutale", "1000"], 302),
    ("husqvarna", ["vitpilen", "701"], 185), ("husqvarna", ["svartpilen", "701"], 185),
    ("royal enfield", ["himalayan"], 130), ("royal enfield", ["meteor"], 120),
]

if 'brand' in df.columns and ('brzina_kmh' not in df.columns or df['brzina_kmh'].isna().any()):
    if 'brzina_kmh' not in df.columns:
        df['brzina_kmh'] = np.nan
    if 'model_naziv' not in df.columns:
        df['model_naziv'] = ''

    def matchuj(row):
        if pd.notna(row.get('brzina_kmh')) and row['brzina_kmh'] > 0:
            return row.get('brzina_kmh', np.nan)
        mn = str(row.get('model_naziv', '')).lower()
        br = str(row.get('brand', '')).lower()
        for (b, kw, spd) in LOOKUP:
            if b not in br: continue
            if all(k in mn for k in kw):
                return float(spd)
        return row.get('brzina_kmh', np.nan)

    df['brzina_kmh'] = df.apply(matchuj, axis=1)
    pok = df['brzina_kmh'].notna().mean()
    print(f"Pokrivenost brzina_kmh nakon dopune: {pok:.1%}")


## 4.4 Imputacija nedostajućih vrijednosti
Numeričke varijable popunjavamo **medijanom** unutar svake kategorije (category-aware imputation).
Kategorijske varijable popunjavamo **najčešćom vrijednošću** (modom).


In [ ]:
num_vars = [c for c in ['kubikaza_ccm','snaga_hp','moment_nm','tezina_kg',
                         'brzina_kmh','rezervoar_l','sjediste_mm',
                         'meduosovinski_mm','broj_brzina','cilindri',
                         'ccm_iz_naziva'] if c in df.columns]

# Koristimo globalni medijan (iz train seta) — ovdje za ilustraciju koristimo cijeli dataset
for col in num_vars:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    med = df[col].median()
    df[col] = df[col].fillna(med)
    print(f"  {col:25} medijan={med:.1f}")

# Kategorijske
if 'hladjenje' in df.columns:
    df['hladjenje'] = df['hladjenje'].fillna(df['hladjenje'].mode()[0])
if 'dvotaktni_flag' in df.columns:
    df['dvotaktni'] = (df['dvotaktni_flag'].astype(str).str.lower().str.contains('two|2-stroke|dvotaktni')).astype(int)
else:
    df['dvotaktni'] = 0

print("\nImputacija završena.")
print(df[num_vars].isnull().sum().to_frame('preostali_nan').T)


## 4.5 Enkodiranje kategorijskih varijabli

In [ ]:
# Enkodiranje ciljne varijable (Category)
le_cat = LabelEncoder()
df['category_enc'] = le_cat.fit_transform(df['Category'])
print(f"Kategorije (18): {list(le_cat.classes_)}")

# One-hot enkodiranje hlađenja
if 'hladjenje' in df.columns:
    df = pd.get_dummies(df, columns=['hladjenje'], prefix='hladjenje', drop_first=False)
    hl_cols = [c for c in df.columns if c.startswith('hladjenje_')]
    print(f"\nHlađenje (one-hot): {hl_cols}")
else:
    hl_cols = []

print(f"\nShape nakon enkodiranja: {df.shape}")


## 4.6 Odabir feature-a i priprema za trening

In [ ]:
# Feature varijable (bez izvedene multikolinearnosti)
feature_cols = [c for c in num_vars if c != 'snaga_po_kg'] + hl_cols + ['dvotaktni']
feature_cols = [c for c in feature_cols if c in df.columns]

X = df[feature_cols].copy()
y = df['category_enc'].values

print(f"Broj feature-a: {len(feature_cols)}")
print(f"Feature-i: {feature_cols}")
print(f"\nBroj uzoraka: {len(X)}")
print(f"Raspodjela klasa (top 5):\n{pd.Series(y).value_counts().head()}")


## 4.7 Feature selection — ANOVA F-test i Mutual Information

In [ ]:
f_scores, _ = f_classif(X, y)
mi_scores = mutual_info_classif(X, y, random_state=42)

sel_df = pd.DataFrame({
    'Feature': feature_cols,
    'ANOVA F': f_scores.round(1),
    'Mutual Info': mi_scores.round(4)
}).sort_values('ANOVA F', ascending=False)

print(sel_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sel_df.sort_values('ANOVA F').plot(kind='barh', x='Feature', y='ANOVA F',
                                    ax=axes[0], color='steelblue', legend=False)
axes[0].set_title('ANOVA F-score (veći = važniji)')
sel_df.sort_values('Mutual Info').plot(kind='barh', x='Feature', y='Mutual Info',
                                        ax=axes[1], color='seagreen', legend=False)
axes[1].set_title('Mutual Information (veći = važniji)')
plt.tight_layout()
plt.show()


## 4.8 Train/test split i skaliranje

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train set: {X_train_sc.shape}")
print(f"Test set:  {X_test_sc.shape}")
print(f"\nRaspodjela klasa u train setu (top 5):")
print(pd.Series(y_train).value_counts().head())


# 5. Modeling and Evaluation — Klasifikacija
Klasifikacija je **nadzorovano učenje** — model uči na osnovu poznatih kategorija (labela).
Koristimo dva algoritma: **Random Forest** (ansambl stabala odluke) i **Logistička regresija**.


## 5.1 Random Forest
Random Forest je ansambl algoritam koji gradi više stabala odluke na nasumičnim podskupovima podataka i glasanjem određuje finalnu predikciju.
Prednosti: robustan na preprilagođavanje, ne zahtijeva skaliranje, daje važnost varijabli.


In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=None,
                             min_samples_leaf=2, random_state=42, n_jobs=-1)
rf.fit(X_train_sc, y_train)
y_pred_rf = rf.predict(X_test_sc)

print("=== Random Forest — Classification Report ===")
print(classification_report(y_test, y_pred_rf,
      target_names=le_cat.classes_, zero_division=0))


In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(13, 11))
cm = confusion_matrix(y_test, y_pred_rf)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le_cat.classes_)
disp.plot(ax=ax, xticks_rotation=45, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Random Forest', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Feature importance
imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
imp.head(12).plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Važnost varijabli — Random Forest (top 12)')
ax.set_ylabel('Importance')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()
print(imp.head(10).to_string())


## 5.2 Logistička regresija
Logistička regresija je linearni model koji predviđa vjerovatnoću klase putem sigmoidne funkcije.
Brža je i interpretabilnija od Random Foresta, ali pretpostavlja linearnu separabilnost.


In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42, C=1.0, solver='lbfgs',
                         n_jobs=-1)
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)

print("=== Logistička regresija — Classification Report ===")
print(classification_report(y_test, y_pred_lr,
      target_names=le_cat.classes_, zero_division=0))


In [ ]:
fig, ax = plt.subplots(figsize=(13, 11))
cm_lr = confusion_matrix(y_test, y_pred_lr)
ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=le_cat.classes_).plot(
    ax=ax, xticks_rotation=45, colorbar=False, cmap='Oranges')
ax.set_title('Confusion Matrix — Logistička regresija', fontsize=13)
plt.tight_layout()
plt.show()


## 5.3 Unakrsna validacija (Cross-validation)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores_rf = cross_val_score(rf, X_train_sc, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)
scores_lr = cross_val_score(lr, X_train_sc, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)

print(f"Random Forest   — CV F1 (macro): {scores_rf.mean():.4f} ± {scores_rf.std():.4f}")
print(f"Logist. regr.   — CV F1 (macro): {scores_lr.mean():.4f} ± {scores_lr.std():.4f}")

comp = pd.DataFrame({
    'Model': ['Random Forest', 'Logistička regresija'],
    'CV F1 (mean)': [scores_rf.mean(), scores_lr.mean()],
    'CV F1 (std)':  [scores_rf.std(),  scores_lr.std()],
})
print("\n" + comp.to_string(index=False))


# 6. Clustering — Klasteriranje
Klasteriranje je **nenadzorovano učenje** — algoritam sam pronalazi prirodne grupe (klastere) bez korištenja poznatih kategorija.
Primjenjujemo četiri metode: K-Means, K-Medoids, K-Modes i K-Prototypes.


## 6.1 K-Means klasteriranje
K-Means dijeli podatke u k klastera minimiziranjem sume kvadratnih udaljenosti od centroida.
**Elbow metoda** pomaže odabiru optimalnog k.


In [ ]:
num_vars_cl = [c for c in ['kubikaza_ccm','snaga_hp','moment_nm','tezina_kg',
                             'brzina_kmh','rezervoar_l','sjediste_mm',
                             'meduosovinski_mm','broj_brzina','cilindri'] if c in df.columns]

df_cl = df[num_vars_cl + ['Category']].copy()
for col in num_vars_cl:
    df_cl[col] = df_cl[col].fillna(df_cl[col].median())

df_s = df_cl.sample(n=min(5000, len(df_cl)), random_state=42)
y_true = df_s['Category'].values
sc2 = StandardScaler()
X_cl = sc2.fit_transform(df_s[num_vars_cl].values)

# Elbow + Silhouette
inertias, sil_scores = [], []
K_range = range(2, 12)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_cl)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_cl, lbl))
    print(f"  k={k}: inertia={km.inertia_:.0f}, sil={sil_scores[-1]:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(list(K_range), inertias, 'o-', color='steelblue', lw=2)
axes[0].set_title('Elbow metoda (inertia)'); axes[0].set_xlabel('k')
axes[0].axvline(x=6, color='red', ls='--', lw=1.5, label='k=6'); axes[0].legend()
axes[1].plot(list(K_range), sil_scores, 'o-', color='seagreen', lw=2)
axes[1].set_title('Silhouette score'); axes[1].set_xlabel('k')
opt_k = list(K_range)[sil_scores.index(max(sil_scores))]
axes[1].axvline(x=opt_k, color='red', ls='--', lw=1.5, label=f'optimum k={opt_k}'); axes[1].legend()
plt.suptitle('Odabir optimalnog k za K-Means', fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
# K-Means k=6 + PCA vizualizacija
km6 = KMeans(n_clusters=6, random_state=42, n_init=10)
y_km6 = km6.fit_predict(X_cl)

sil_km = silhouette_score(X_cl, y_km6)
db_km  = davies_bouldin_score(X_cl, y_km6)
ch_km  = calinski_harabasz_score(X_cl, y_km6)
ari_km = adjusted_rand_score(y_true, y_km6)
print(f"K-Means k=6: Silhouette={sil_km:.3f}, DB={db_km:.3f}, CH={ch_km:.0f}, ARI={ari_km:.3f}")

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_cl)
var_exp = pca.explained_variance_ratio_

boje6 = plt.cm.Set1(np.linspace(0, 1, 6))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cats = sorted(set(y_true))
pal = plt.cm.tab20(np.linspace(0, 1, len(cats)))
for i, cat in enumerate(cats):
    m = y_true == cat
    axes[0].scatter(X_pca[m,0], X_pca[m,1], c=[pal[i]], label=cat, s=4, alpha=0.5)
axes[0].set_title(f'Stvarne kategorije (18)\nPC1={var_exp[0]*100:.1f}%, PC2={var_exp[1]*100:.1f}%')
axes[0].legend(fontsize=4, markerscale=3, ncol=2)
for ki in range(6):
    m = y_km6 == ki
    axes[1].scatter(X_pca[m,0], X_pca[m,1], c=[boje6[ki]], label=f'K{ki+1}', s=4, alpha=0.5)
axes[1].set_title(f'K-Means k=6\nSil={sil_km:.3f}, ARI={ari_km:.3f}')
axes[1].legend(fontsize=8, markerscale=3)
plt.tight_layout(); plt.show()


## 6.2 K-Medoids (PAM)
K-Medoids je robusnija varijanta K-Meansa — umjesto imaginarnog prosjeka, centar klastera je **stvarni motor** iz dataseta (medoid).
Implementiramo PAM algoritam ručno jer sklearn nema stabilnu implementaciju za naš Python.


In [ ]:
def kmedoids_pam(X, k, max_iter=100, random_state=42):
    np.random.seed(random_state)
    n = len(X)
    medoid_idx = np.random.choice(n, k, replace=False)
    for _ in range(max_iter):
        dists = np.array([[np.linalg.norm(X[i]-X[m]) for m in medoid_idx] for i in range(n)])
        labels = np.argmin(dists, axis=1)
        new_med = medoid_idx.copy()
        for ki in range(k):
            mask = labels == ki
            if mask.sum() == 0: continue
            idx = np.where(mask)[0]
            sub = X[mask]
            intra = np.sum(np.linalg.norm(sub[:,None]-sub[None,:], axis=2), axis=1)
            new_med[ki] = idx[np.argmin(intra)]
        if np.all(new_med == medoid_idx): break
        medoid_idx = new_med
    return labels, medoid_idx

df_kmed = df_s.sample(n=min(800, len(df_s)), random_state=42)
X_kmed  = sc2.transform(df_kmed[num_vars_cl].values)
y_true_kmed = df_kmed['Category'].values

print("Treniram K-Medoids (PAM, k=6) na 800 uzoraka...")
labels_kmed, medoids = kmedoids_pam(X_kmed, k=6)

sil_kmed = silhouette_score(X_kmed, labels_kmed)
db_kmed  = davies_bouldin_score(X_kmed, labels_kmed)
ch_kmed  = calinski_harabasz_score(X_kmed, labels_kmed)
ari_kmed = adjusted_rand_score(y_true_kmed, labels_kmed)
print(f"K-Medoids k=6: Silhouette={sil_kmed:.3f}, DB={db_kmed:.3f}, CH={ch_kmed:.0f}, ARI={ari_kmed:.3f}")

print("\nMedoidi — stvarni motori koji su centri klastera:")
df_kmed_r = df_kmed.reset_index(drop=True)
for ki, mid in enumerate(medoids):
    row = df_kmed_r.iloc[mid]
    print(f"  Klaster {ki+1}: {row['kubikaza_ccm']:.0f} ccm, {row.get('snaga_hp',0):.0f} HP | {row['Category']}")


In [ ]:
X_pca_kmed = pca.transform(X_kmed)
fig, ax = plt.subplots(figsize=(8, 5))
for ki in range(6):
    m = labels_kmed == ki
    ax.scatter(X_pca_kmed[m,0], X_pca_kmed[m,1], c=[boje6[ki]], label=f'K{ki+1}', s=8, alpha=0.6)
for ki, mid in enumerate(medoids):
    ax.scatter(X_pca_kmed[mid,0], X_pca_kmed[mid,1], c='black', marker='*', s=250, zorder=5)
ax.set_title(f'K-Medoids / PAM (k=6) — ★ = medoid (stvarni motor)\nSil={sil_kmed:.3f}, ARI={ari_kmed:.3f}')
ax.legend(fontsize=8); plt.tight_layout(); plt.show()


## 6.3 K-Modes klasteriranje
K-Modes radi isključivo s kategorijskim podacima koristeći **mod** (najčešću vrijednost) umjesto prosjeka.
Numeričke varijable moraju biti pretvorene u kategorije (binning).


In [ ]:
df_km3 = df_s.copy()
df_km3['kub_bin']   = pd.cut(df_km3['kubikaza_ccm'],
    bins=[0,125,300,600,1000,10000], labels=['micro','mali','srednji','veliki','ekstrem'])
df_km3['snaga_bin'] = pd.cut(df_km3.get('snaga_hp', pd.Series(dtype=float)),
    bins=[0,20,50,100,200,1000],     labels=['slaba','niska','srednja','visoka','ekstremna'])
df_km3['tezina_bin']= pd.cut(df_km3.get('tezina_kg', pd.Series(dtype=float)),
    bins=[0,80,130,180,300,1000],    labels=['lagan','normalan','tezak','veoma_tezak','ekstrem'])
df_km3['brzina_bin']= pd.cut(df_km3.get('brzina_kmh', pd.Series(dtype=float)),
    bins=[0,80,130,180,250,700],     labels=['spor','umjeren','brz','veoma_brz','superbrz'])

cat_cols_km = ['kub_bin','snaga_bin','tezina_bin','brzina_bin','dvotaktni']
cat_cols_km = [c for c in cat_cols_km if c in df_km3.columns]
for c in cat_cols_km:
    if hasattr(df_km3[c], 'cat'):
        df_km3[c] = df_km3[c].cat.add_categories('nepoznato').fillna('nepoznato')
    else:
        df_km3[c] = df_km3[c].fillna('nepoznato')

X_modes = df_km3[cat_cols_km].astype(str).values
km6_modes = KModes(n_clusters=6, init='Huang', n_init=3, random_state=42)
y_modes = km6_modes.fit_predict(X_modes)

sil_modes = silhouette_score(X_cl[:len(df_km3)], y_modes)
db_modes  = davies_bouldin_score(X_cl[:len(df_km3)], y_modes)
ch_modes  = calinski_harabasz_score(X_cl[:len(df_km3)], y_modes)
ari_modes = adjusted_rand_score(df_s['Category'].values[:len(df_km3)], y_modes)
print(f"K-Modes k=6: Silhouette={sil_modes:.3f}, DB={db_modes:.3f}, CH={ch_modes:.0f}, ARI={ari_modes:.3f}")


## 6.4 K-Prototypes klasteriranje
K-Prototypes kombinuje K-Means (za numeričke) i K-Modes (za kategorijske) varijable — idealan za mješovite datasetove.


In [ ]:
df_kp = df_s.sample(n=min(2000, len(df_s)), random_state=42).copy()
if 'dvotaktni' not in df_kp.columns:
    df_kp['dvotaktni'] = 0
X_num_kp = StandardScaler().fit_transform(df_kp[num_vars_cl].values)
X_cat_kp = df_kp[['dvotaktni']].astype(str).values
X_proto  = np.hstack([X_num_kp, X_cat_kp])
cat_idx  = [X_num_kp.shape[1]]

kp6 = KPrototypes(n_clusters=6, init='Cao', n_init=3, random_state=42)
y_kp6 = kp6.fit_predict(X_proto, categorical=cat_idx)
y_true_kp = df_kp['Category'].values

sil_kp = silhouette_score(X_num_kp, y_kp6)
db_kp  = davies_bouldin_score(X_num_kp, y_kp6)
ch_kp  = calinski_harabasz_score(X_num_kp, y_kp6)
ari_kp = adjusted_rand_score(y_true_kp, y_kp6)
print(f"K-Prototypes k=6: Silhouette={sil_kp:.3f}, DB={db_kp:.3f}, CH={ch_kp:.0f}, ARI={ari_kp:.3f}")


## 6.5 Usporedba svih metoda klasteriranja

In [ ]:
rezultati = pd.DataFrame([
    ['K-Means',      sil_km,   db_km,   ch_km,   ari_km,   'Numerički', 'Prosjek (mean)'],
    ['K-Medoids',    sil_kmed, db_kmed, ch_kmed, ari_kmed, 'Numerički', 'Medoid (stvarni motor)'],
    ['K-Modes',      sil_modes,db_modes,ch_modes,ari_modes,'Kategorijski','Mod (najčešća vr.)'],
    ['K-Prototypes', sil_kp,   db_kp,   ch_kp,   ari_kp,   'Mješoviti', 'Prosjek + mod'],
], columns=['Metoda','Silhouette ↑','DB ↓','CH ↑','ARI ↑','Tip','Centar klastera'])

print(rezultati.to_string(index=False))

# Vizualizacija metrika
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
metrics = [('Silhouette ↑','steelblue'), ('DB ↓','coral'), ('CH ↑','seagreen'), ('ARI ↑','purple')]
for ax, (met, col) in zip(axes, metrics):
    vals = rezultati[met].values
    bars = ax.bar(rezultati['Metoda'], vals, color=col, alpha=0.8)
    ax.set_title(met); ax.tick_params(axis='x', rotation=30)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01*max(abs(v) for v in vals),
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)
plt.suptitle('Usporedba metoda klasteriranja — sve metrike (k=6)', fontsize=12)
plt.tight_layout(); plt.show()


# 7. Zaključak
Sprovedena je kompletna analiza mašinskog učenja na datasetu od **38.472 motocikala** sa **18 kategorija**.

**Klasifikacija (nadzorovano učenje):**
- **Random Forest** postiže F1 macro ≈ 0,67, accuracy ≈ 0,78 — best rezultat zahvaljujući robusnosti na outliere i nelinearnim vezama između tehničkih specifikacija
- **Logistička regresija** postiže F1 ≈ 0,32 — slabija jer pretpostavlja linearnu separabilnost klasa, a granice između 18 kategorija su nelinearne

**Klasteriranje (nenadzorovano učenje):**
- **K-Means** daje najbolje metrike (Silhouette≈0,21, ARI≈0,13) — optimalan za pretežno numerički dataset
- **K-Medoids** je robusniji na ekstreme, daje stvarne motore kao centre klastera (interpretabilnije)
- **K-Modes** ima negativan Silhouette (≈−0,08) jer diskretizacija (binning) gubi preciznost
- **K-Prototypes** je kompromis između K-Meansa i K-Modesa za mješovite podatke

**Opći zaključak:** Nadzorovani Random Forest (F1≈0,67) daleko nadmašuje bilo koje klasteriranje (ARI≤0,13) jer koristi informaciju o kategorijama pri treniranju. Klasteriranje otkriva 2–6 prirodnih profila motocikala po tehničkim karakteristikama, ali ne može rekonstruirati točno 18 kategorija bez supervizije.


# 8. Ručna predikcija — unesi specifikacije motora
Unesi tehničke podatke motocikla i Random Forest će predvidjeti kojoj kategoriji pripada.

In [ ]:
# ============================================================
# UNESI SPECIFIKACIJE MOTORA OVDJE:
# ============================================================
kubikaza_ccm     = 1000   # ccm (kubikaza motora)
snaga_hp         = 100    # HP (konjske snage)
moment_nm        = 100    # Nm (obrtni moment)
tezina_kg        = 200    # kg (suha težina)
brzina_kmh       = 250    # km/h (max brzina)
rezervoar_l      = 17     # litara (rezervoar)
sjediste_mm      = 820    # mm (visina sjedišta)
meduosovinski_mm = 1500   # mm (međuosovinski razmak)
broj_brzina      = 6      # broj brzina
ccm_iz_naziva    = 0      # ostavi 0 ako ne znaš
hladjenje_tip    = 'Liquid'  # 'Air', 'Liquid' ili 'Oil & air'
# ============================================================

import numpy as np, pandas as pd

unos = {
    'kubikaza_ccm': kubikaza_ccm,
    'snaga_hp': snaga_hp,
    'moment_nm': moment_nm,
    'tezina_kg': tezina_kg,
    'brzina_kmh': brzina_kmh,
    'rezervoar_l': rezervoar_l,
    'sjediste_mm': sjediste_mm,
    'meduosovinski_mm': meduosovinski_mm,
    'broj_brzina': broj_brzina,
    'ccm_iz_naziva': ccm_iz_naziva,
    'dvotaktni': 0,
}

# Dodaj one-hot hlađenje
for col in feature_cols:
    if col.startswith('hladjenje_'):
        tip = col.replace('hladjenje_', '')
        unos[col] = 1 if tip == hladjenje_tip else 0

red = pd.DataFrame([unos])
red = red.reindex(columns=feature_cols, fill_value=0)
red_sc = scaler.transform(red)

predikcija = rf.predict(red_sc)[0]
vjerovatnoce = rf.predict_proba(red_sc)[0]
kategorija = le_cat.inverse_transform([predikcija])[0]

print(f'\n>>> PREDVIĐENA KATEGORIJA: {kategorija} <<<\n')
print('Top 5 vjerovatnoća:')
top5 = sorted(zip(le_cat.classes_, vjerovatnoce), key=lambda x: -x[1])[:5]
for kat, prob in top5:
    bar = '█' * int(prob * 30)
    print(f'  {kat:30} {bar} {prob*100:.1f}%')
